# SIH 26078: GPU training on Colab / Kaggle

This notebook runs the same training scripts as local CPU, unchanged. They pick the device via `pipeline/compute.py` (`cuda` if present). Data is a **SYNTHETIC** training bundle (`training_bundle.zip`, <2 GB) made locally with `python scripts/package_training_data.py`.

Steps: 1) check GPU, 2) get code, 3) upload/unpack the bundle, 4) install, 5) train, 6) download checkpoints and metrics back to the repo (`models/`).

In [ ]:
!nvidia-smi || echo 'no GPU - runtime > change runtime type > GPU'

## 1. Code
Either clone your pushed repo, or upload a zip of the repo (without `data/`).

In [ ]:
import os
REPO_URL = ''  # e.g. 'https://github.com/<you>/<repo>.git' (leave empty to upload a zip)
if REPO_URL:
    !git clone -q $REPO_URL repo
else:
    from google.colab import files  # on Kaggle: add the zip as a dataset instead
    up = files.upload()             # upload repo.zip
    !unzip -q -o repo.zip -d repo
%cd repo

## 2. Training bundle
Upload `training_bundle.zip` (Colab: file upload or Google Drive; Kaggle: attach as a dataset at `/kaggle/input/...`).

In [ ]:
import glob, zipfile
cands = glob.glob('/kaggle/input/*/training_bundle.zip') + glob.glob('/content/drive/MyDrive/training_bundle.zip')
if not cands:
    from google.colab import files
    files.upload()
    cands = ['training_bundle.zip']
zipfile.ZipFile(cands[0]).extractall('.')
!ls -la bundle && cat bundle/manifest.json | head -30

## 3. Install
Colab/Kaggle already ship a CUDA torch; only the light dependencies are added.

In [ ]:
!pip -q install xarray netCDF4 scipy scikit-learn pandas matplotlib
import torch; print(torch.__version__, torch.cuda.is_available())
!python -m pipeline.compute

## 4. Train
Same commands as locally; `SIH_DEVICE=cuda` is picked up automatically when a GPU exists. Seeds are fixed. Splits are whole cases from `pipeline/splits.py`.

In [ ]:
# GNN tracker (Phase 2) - bundle/ contains the object graphs (data/graphs/*.npz)
for v in ['full', 'temporal', 'cross', 'none']:
    !python scripts/train_tracker.py --graphs bundle --out models/tracker --variant {v}

In [ ]:
# Downscalers (Phase 3): U-Net and CorrDiff-style residual diffusion
!python scripts/train_downscaler.py --bundle bundle --model unet --out models/unet
!python scripts/train_downscaler.py --bundle bundle --model diffusion --out models/diffusion --epochs 60

## 5. Download results
Copy `models/` back into the local repo, then run `make baseline`-style evaluation locally on the full-precision NetCDF data.

In [ ]:
!zip -q -r models.zip models
try:
    from google.colab import files; files.download('models.zip')
except ImportError:
    print('Kaggle: models.zip is in /kaggle/working - download from the Output tab')